# Cluster Genérico - K-means

In [0]:
json_cluster = [
{
"cluster_id": 0,
"nome": "cluster_generico_kmeans",
"tabela": "workspace.default.data_lake_hermes_ai.ouro.cluster_1"
"descricao": "Cluster de uso genérico utilizando o modelo k-means, para criação de cluster por similaridade entre variáveis relacionadas ao negócio do cliente, como serviços totvs utilizados, ramo de atividade do cliente, modelo de negócio e contrato, segmento de atuação etc. Esse cluster tem como objetivo trazer clientes próximos em relação a atuação e o serviço totvs utilizado para facilitar uma possível análise de clientes similares em sua atuação e que usam produtos e serviços da totvs similares, possibilitam uma tratativa de atendimento personalizada com base no negócio do cliente",
"conceito": "Este modelo agrupa os dados de forma que os pontos dentro de um mesmo cluster sejam o mais semelhantes possível, ao mesmo tempo que os clusters sejam o mais diferentes possível uns dos outros."
},
]

In [0]:
%sql
SELECT 
* 
FROM data_lake_hermes_ai.ouro.clusters_catalogo

## Base de dados

In [0]:
%sql
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.base_cluster_generico AS (
SELECT 
a.CD_CLIENTE,
a.UF,
a.regiao,
a.is_pessoa_fisica,
a.DS_SEGMENTO,
a.DS_SUBSEGMENTO,
a.categoria_faturamento,
a.MARCA_TOTVS,
a.MODAL_COMERC,
a.PERIODICIDADE,
a.SITUACAO_CONTRATO,
a.Tempo_relacionamento,
CASE 
    WHEN b.grupo_nps IS NOT NULL THEN b.grupo_nps
    WHEN b.grupo_nps IS NULL THEN 'sem_nps'
    ELSE 'sem_nps'
END AS nps_aquisicao,
CASE 
    WHEN c.grupo_nps IS NOT NULL THEN c.grupo_nps
    WHEN c.grupo_nps IS NULL THEN 'sem_nps'
    ELSE 'sem_nps'
END AS nps_implantacao,
CASE 
    WHEN e.grupo_nps IS NOT NULL THEN e.grupo_nps
    WHEN e.grupo_nps IS NULL THEN 'sem_nps'
    ELSE 'sem_nps'
END AS nps_produto,
CASE 
    WHEN f.grupo_nps IS NOT NULL THEN f.grupo_nps
    WHEN f.grupo_nps IS NULL THEN 'sem_nps'
    ELSE 'sem_nps'
END AS nps_relacional,
CASE 
    WHEN g.grupo_nps IS NOT NULL THEN g.grupo_nps
    WHEN g.grupo_nps IS NULL THEN 'sem_nps'
    ELSE 'sem_nps'
END AS nps_suporte
FROM data_lake_hermes_ai.prata.validation_totvs a
LEFT JOIN data_lake_hermes_ai.prata.nps_aquisicao_consolidado b ON a.CD_CLIENTE = b.Cod_Cliente
LEFT JOIN data_lake_hermes_ai.prata.nps_implantacao_consolidado c  ON a.CD_CLIENTE = c.Cod_Cliente
LEFT JOIN data_lake_hermes_ai.prata.nps_onboarding d ON a.CD_CLIENTE = d.Cod_Cliente
LEFT JOIN data_lake_hermes_ai.prata.nps_produto e ON a.CD_CLIENTE = e.Cod_Cliente
LEFT JOIN data_lake_hermes_ai.prata.nps_relacional_consolidado f ON a.CD_CLIENTE = f.Cod_Cliente
LEFT JOIN data_lake_hermes_ai.prata.nps_suporte_consolidado g ON a.CD_CLIENTE = g.Cod_Cliente)

## Pré Processamento e Ajuste das Variáveis

In [0]:
import pandas as pd

table_name = 'data_lake_hermes_ai.prata.base_cluster_generico'
# ONE HOT ENCODING
df_raw = spark.read.table(table_name).toPandas()
df_encoded = pd.get_dummies(
    df_raw,
    columns=['UF', 'regiao', 'DS_SEGMENTO','DS_SUBSEGMENTO','categoria_faturamento','MARCA_TOTVS','MODAL_COMERC','PERIODICIDADE','SITUACAO_CONTRATO','nps_aquisicao','nps_implantacao','nps_produto','nps_relacional','nps_suporte'],
    prefix=['is_', 'is_', 'is_','is_', 'is_', 'is_','is_', 'is_', 'is_','is_', 'is_', 'is_', 'is_','is_']
)

df_encoded

## Cálculo de k

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Seleciona amostra
df_sample = df_encoded.head(70000)

# Remove duplicatas na lista de features
features = list(dict.fromkeys([
    'is_pessoa_fisica','Tempo_relacionamento','is__AC','is__AL','is__AM','is__AP','is__BA','is__CE',
    'is__DF','is__ES','is__GO','is__MA','is__MG','is__MS','is__MT','is__PA','is__PB','is__PE','is__PI',
    'is__PR','is__RJ','is__RN','is__RO','is__RR','is__RS','is__SC','is__SE','is__SP','is__TO',
    'is__Centro-Oeste','is__Nordeste','is__Norte','is__Sudeste','is__Sul','is__AGROINDUSTRIA',
    'is__CONSTRUCAO E PROJETOS','is__DISTRIBUICAO','is__EDUCACIONAL','is__FINANCIAL SERVICES',
    'is__MODALIDADE SERVICOS NÃO RECORRENTES','is__MODALIDADE SERVICOS RECORRENTES',
    'is__MODALIDADE TRADICIONAL','is__MPN','is__OUTROS','is__SERIE 3','is__SERVIÇOS',
    'is__00 - Mensal','is__01 - Bimestral','is__02 - Trimestral','is__03 - Quadrimestral',
    'is__05 - Semestral','is__11 - Anual','is__ATIVO','is__CANCELADO','is__FATURAR',
    'is__GRATUITO','is__PENDENTE','is__SUSPENSO','is__TROCADO','is__detrator','is__passivo',
    'is__promotor','is__sem_nps'
]))

X = df_sample[features]

# Normalização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Testa vários k
ks = range(2, 11)
inertias = []
sil_scores = []

for k in ks:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=50,max_iter=500,algorithm='elkan', random_state=123)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Plot Elbow e Silhouette
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(ks, inertias, '-o')
plt.title('Elbow (Inertia)')
plt.xlabel('k')
plt.ylabel('Inertia (WCSS)')

plt.subplot(1,2,2)
plt.plot(ks, sil_scores, '-o', color='orange')
plt.title('Silhouette Score')
plt.xlabel('k')
plt.ylabel('Score')
plt.tight_layout()
plt.show()

## K-means

In [0]:
k = 3
kmeans = KMeans(n_clusters=k, init='k-means++', n_init=50,max_iter=500,algorithm='elkan', random_state=123)

# Ajusta e obtém os rótulos
labels = kmeans.fit_predict(X_scaled)

df_clustered = df_sample.copy()
df_clustered['cluster'] = labels

df_clustered_v1 = df_clustered[['CD_CLIENTE', 'cluster']]

df_clustered_v1

## Criação de tabela com saída do modelo

In [0]:
# Se df_clustered for um pandas DataFrame, converta para Spark
spark_df = spark.createDataFrame(df_clustered_v1)

# Gravar na área de tabelas gerenciadas do Databricks
spark_df.write.mode("overwrite").saveAsTable("data_lake_hermes_ai.ouro.cluster_1")

## Visualização dos resultados

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.7)
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.title("Distribuição dos Clusters (PCA 2D)")
plt.colorbar(label='Cluster')
plt.show()

In [0]:
%sql
SELECT 
*
FROM data_lake_hermes_ai.ouro.cluster_1